# 03 · YOLO fine-tuning (Colab)

Phase 3 of `notes/plan.md`: train stronger Live/Dead detectors than baseline B7 (YOLO26n, default settings) and compare them with the same count metrics on `splits/splits.csv`.

1. Pick experiments in **Experiments**: model size and Ultralytics training arguments (augmentation, epochs, ...).
2. **Train**: each run saves to `outputs/yolo/<name>/` on Drive. If Colab disconnects, rerun the notebook and it **resumes from `last.pt`**; finished runs are skipped.
3. **Evaluate**: the confidence threshold is tuned on **valid** to minimize total-count MAE, then each run is scored on **test**. Results go to `outputs/results/yolo_finetune.csv`.
4. Optional: leave-one-session-out for the best run, worst-image review, and inference time.

Run `00_setup_and_download.ipynb` first and use a GPU runtime (T4 or better).

In [ ]:
# --- Colab bootstrap (identical in every notebook in this folder, except EXTRAS) ---
# Expects the contents of notebooks/notebooks_colab_draft/ uploaded to DRIVE_ROOT on Google Drive:
# pyproject.toml, src/cellcount/, splits/splits.csv and these notebooks (see README.md).
# data/ and outputs/ are created there too, so they survive runtime resets.
# Secrets (key icon in the left sidebar, notebook access on): ROBOFLOW_EXPORT_URL.
import os, shutil, subprocess, sys
from pathlib import Path

DRIVE_ROOT = "/content/drive/MyDrive/cell-image-counting"
EXTRAS = "[model]"  # "[model]" adds torch, ultralytics and cellpose

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    root = Path(DRIVE_ROOT)
    missing = [p for p in ("pyproject.toml", "src/cellcount/__init__.py", "splits/splits.csv") if not (root / p).exists()]
    if missing:
        raise FileNotFoundError(f"missing in {root}: {missing}. Upload notebooks_colab_draft/ there (README.md).")
    # Install from a local copy: Drive is slow and pip would write build files into it.
    # Re-copied on every run, so re-uploaded code is picked up after a runtime restart.
    code_dir = Path("/content/cellcount_code")
    shutil.rmtree(code_dir, ignore_errors=True)
    shutil.copytree(root / "src", code_dir / "src", ignore=shutil.ignore_patterns("__pycache__", "*.egg-info"))
    shutil.copy(root / "pyproject.toml", code_dir)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{code_dir}{EXTRAS}"], check=True)
    sys.path.insert(0, str(code_dir / "src"))  # import without restarting the runtime
    os.environ["CELLCOUNT_STORAGE"] = DRIVE_ROOT  # data/, outputs/ and splits/ live here
    os.chdir("/content")  # pretrained weights (yolo26n.pt, ...) download here, not to Drive

In [ ]:
import torch

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU: Runtime > Change runtime type > T4 GPU (or better). CPU training takes many hours.")

In [ ]:
import shutil
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Image, display

from cellcount import viz, yolo
from cellcount.config import get_paths
from cellcount.data import RAW_DIRNAME, load_coco
from cellcount.eda import count_bins
from cellcount.metrics import count_metrics, count_metrics_by, detection_metrics, per_image_counts
from cellcount.splits import leave_one_source_out, load_splits

sns.set_theme(style="whitegrid", context="notebook")
paths = get_paths()
RUNS = paths.outputs / "yolo"
FIG = paths.outputs / "figures" / "yolo_finetune"
FIG.mkdir(parents=True, exist_ok=True)
(paths.outputs / "results").mkdir(exist_ok=True)

# Copy the images to the runtime's local disk: Drive reads are slow, and export_yolo makes
# symlinks, which Drive doesn't support. Results and weights still go to Drive (paths.outputs).
WORK = Path("/content/work") if IN_COLAB else paths.data
if not (WORK / RAW_DIRNAME).exists():
    shutil.copytree(paths.data / RAW_DIRNAME, WORK / RAW_DIRNAME)
images, boxes = load_coco(WORK / RAW_DIRNAME)
split_df = load_splits(paths.root / "splits" / "splits.csv")
truth = per_image_counts(boxes, images["uid"]).rename(columns={"live": "live_true", "dead": "dead_true"})
df = (images[["uid", "path", "width", "height", "source"]]
      .merge(split_df[["uid", "split"]], on="uid")
      .merge(truth, on="uid"))
df["total_true"] = df["live_true"] + df["dead_true"]
df["count_bin"] = count_bins(df["total_true"])
train, valid, test = (df[df["split"] == s] for s in ("train", "valid", "test"))
data_yaml = yolo.export_yolo(images, boxes, split_df, WORK / "yolo")
print(df["split"].value_counts().to_dict(), "| sessions:", df["source"].nunique())

## Experiments

Each entry is a run name, a pretrained model, and [Ultralytics training arguments](https://docs.ultralytics.com/modes/train/#train-settings) on top of `yolo.train`'s defaults (100 epochs, 640 px, patience 20, seed 0). The `aug` recipe follows `plan.md` §5.1–5.2:

- **Scale and crops:** `scale` zooms in and out by up to ±50%; `mosaic` stitches 4 images, so the model sees cells at the edges and in new layouts. It is turned off for the last `close_mosaic` epochs.
- **Photometric:** brightness and saturation vary a lot between sessions, but **hue stays nearly fixed** (`hsv_h` small), because blue staining is what marks a Dead cell.
- **Geometry:** cells have no orientation, so vertical and horizontal flips are safe. Arbitrary rotations stay off because they loosen axis-aligned boxes.
- **Blur:** Ultralytics applies light blur through Albumentations when it is installed (`pip install albumentations`). Stronger blur tests belong to the Phase 4 stress tests.

Set `RUN` to the experiments to train in this session. On a T4, expect very roughly 15–30 min for `n`, 30–60 min for `s` and 1–2 h for `m` at 100–150 epochs.

In [ ]:
AUG = dict(hsv_h=0.005, hsv_s=0.5, hsv_v=0.4, scale=0.5, translate=0.1,
           flipud=0.5, fliplr=0.5, mosaic=1.0, close_mosaic=15)

EXPERIMENTS = {
    "yolo26n_default": dict(model="yolo26n.pt"),                                   # B7, as in 02
    "yolo26n_aug":     dict(model="yolo26n.pt", epochs=150, patience=30, **AUG),
    "yolo26s_aug":     dict(model="yolo26s.pt", epochs=150, patience=30, **AUG),
    "yolo26m_aug":     dict(model="yolo26m.pt", epochs=150, patience=30, batch=8, **AUG),
}
RUN = ["yolo26n_default", "yolo26n_aug", "yolo26s_aug"]

COMMON = dict(cache="ram", workers=2)  # 500 images fit in RAM; Colab has 2 CPU cores

## Train

A run is skipped once it has finished (a `DONE` marker, or a final `last.pt`). If it was cut off mid-training, it resumes from `last.pt`; otherwise it starts fresh. Delete `outputs/yolo/<name>/` to retrain from scratch.

In [ ]:
import torch
from ultralytics import YOLO


def resumable(last):
    """True for a checkpoint saved mid-training. Ultralytics strips the optimizer when a run
    ends, and resuming such a file silently starts an unrelated training, so check first."""
    ckpt = torch.load(last, map_location="cpu", weights_only=False)
    return ckpt.get("optimizer") is not None and ckpt.get("epoch", -1) >= 0


def train_or_resume(data_yaml, project, name, cfg):
    run_dir = project / name
    best, last = run_dir / "weights" / "best.pt", run_dir / "weights" / "last.pt"
    if (run_dir / "DONE").exists() or (last.exists() and not resumable(last)):
        print(f"{name}: finished, skipping")
    else:
        t0 = time.perf_counter()
        if last.exists():
            print(f"{name}: resuming from {last}")
            YOLO(str(last)).train(resume=True)
        else:
            args = {k: v for k, v in cfg.items() if k != "model"}
            yolo.train(data_yaml, project, model=cfg["model"], name=name, **COMMON, **args)
        print(f"{name}: {(time.perf_counter() - t0) / 60:.1f} min this session")
    (run_dir / "DONE").touch()
    return best


weights = {name: train_or_resume(data_yaml, RUNS, name, EXPERIMENTS[name]) for name in RUN}

In [ ]:
for name, w in weights.items():
    print(name)
    display(Image(filename=str(w.parents[1] / "results.png"), width=900))

## Evaluate on test

Same protocol as B7: predict with a low confidence, tune the threshold on **valid** to minimize total-count MAE, then score **test**. The table also lists the earlier baselines from `outputs/results/baselines.csv` when it is on Drive.

In [ ]:
results, preds, kept = [], {}, {}


def with_counts(frame, pred_boxes):
    c = per_image_counts(pred_boxes, frame["uid"]).rename(columns={"live": "live_pred", "dead": "dead_pred"})
    return frame.merge(c, on="uid")


def evaluate(method, level, fold, counted, pred_boxes, **extra):
    row = {"method": method, "level": level, "fold": fold, **extra, **count_metrics(counted)}
    det = detection_metrics(pred_boxes[pred_boxes["uid"].isin(counted["uid"])],
                            boxes[boxes["uid"].isin(counted["uid"])])
    for cls, m in det.iterrows():
        row.update({f"{k}_{cls.lower()}": m[k] for k in ("precision", "recall", "f1")})
    results.append(row)
    return row


for name, w in weights.items():
    b = yolo.predict(w, pd.concat([valid, test]))
    conf_t = yolo.tune_conf(b[b["uid"].isin(valid["uid"])], valid)
    kept[name] = b[(b["conf"] >= conf_t) & b["uid"].isin(test["uid"])]
    preds[name] = with_counts(test, kept[name])
    evaluate(name, "random split", "test", preds[name], kept[name], conf=conf_t)

res = pd.DataFrame(results)
out_csv = paths.outputs / "results" / "yolo_finetune.csv"
if out_csv.exists():  # keep runs evaluated in earlier sessions
    old = pd.read_csv(out_csv)
    res = pd.concat([old[~old.set_index(["method", "level", "fold"]).index.isin(res.set_index(["method", "level", "fold"]).index)], res])
res.to_csv(out_csv, index=False)

cols = ["conf", "mae_total", "mae_live", "mae_dead", "viability_err_pp", "within_10pct", "r2_total",
        "precision_cell", "recall_cell", "f1_live", "f1_dead"]
table = res[res["fold"] == "test"].set_index("method")
base_csv = paths.outputs / "results" / "baselines.csv"
if base_csv.exists():
    base = pd.read_csv(base_csv)
    table = pd.concat([base[base["fold"] == "test"].set_index("method"), table])
table.reindex(columns=cols).sort_values("mae_total").round(3)

In [ ]:
methods = list(preds)
fig, axes = plt.subplots(2, len(methods), figsize=(4 * len(methods), 8), squeeze=False)
for j, m in enumerate(methods):
    p = preds[m]
    for i, (t, q, label) in enumerate([("total_true", None, "Total"), ("dead_true", "dead_pred", "Dead")]):
        ax = axes[i, j]
        pred = p["live_pred"] + p["dead_pred"] if q is None else p[q]
        sns.scatterplot(x=p[t], y=pred, hue=p["source"], ax=ax, s=25, legend=(i == 0 and j == 0))
        lim = [0, max(p[t].max(), pred.max()) * 1.05]
        ax.plot(lim, lim, "k--", lw=0.8)
        ax.set(xlim=lim, ylim=lim, xlabel=f"true {label}", ylabel=f"predicted {label}", title=f"{m} · {label}")
fig.tight_layout()
fig.savefig(FIG / "01_pred_vs_true.png", dpi=150)

In [ ]:
by = []
for m, p in preds.items():
    for col, name in [("source", "session"), ("count_bin", "count bin")]:
        g = count_metrics_by(p, p[col])[["mae_total"]].reset_index(names="group")
        by.append(g.assign(method=m, by=name, group=g["group"].astype(str)))
by = pd.concat(by)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, name in zip(axes, ["session", "count bin"]):
    sns.barplot(data=by[by["by"] == name], x="group", y="mae_total", hue="method", ax=ax)
    ax.set(title=f"Total-count MAE by {name}", xlabel=name, ylabel="MAE (cells)")
    ax.tick_params(axis="x", rotation=20)
fig.tight_layout()
fig.savefig(FIG / "02_mae_by_session_and_count.png", dpi=150)

## Error analysis: worst test images

Largest total-count errors for the best run (lowest test MAE). Yellow: labelled boxes. Cyan / magenta: predicted Live / Dead. `plan.md` asks for the worst 20 after each experiment; raise `N_WORST` to page through them.

In [ ]:
N_WORST = 6
best_name = min(preds, key=lambda m: count_metrics(preds[m])["mae_total"])
by_uid = df.set_index("uid", drop=False)
p = preds[best_name].assign(err=lambda d: d["live_pred"] + d["dead_pred"] - d["total_true"])
worst = p.reindex(p["err"].abs().sort_values(ascending=False).index).head(N_WORST)
ncols = 3
fig, axes = plt.subplots(-(-N_WORST // ncols), ncols, figsize=(15, 5 * -(-N_WORST // ncols)), squeeze=False)
for ax, r in zip(axes.flat, worst.itertuples()):
    viz.show_prediction(by_uid.loc[r.uid], boxes, kept[best_name], ax=ax,
                        title=f"{r.uid} ({r.source[-3:]})  true {r.total_true}, pred {r.live_pred + r.dead_pred:.0f}")
for ax in axes.flat[len(worst):]:
    ax.axis("off")
fig.suptitle(f"{best_name}: worst {N_WORST} test images")
fig.tight_layout()
fig.savefig(FIG / f"03_worst_{best_name}.png", dpi=120)

## Optional: leave-one-session-out

Retrains one experiment five times, each time holding out one session as test (its valid images stay in valid). This is the real generalization check (Phase 4). Each fold resumes like the main runs, so it survives disconnects. It costs about 5× the run's training time.

In [ ]:
RUN_LOSO = False
LOSO_EXPERIMENT = "yolo26n_aug"

if RUN_LOSO:
    cfg = EXPERIMENTS[LOSO_EXPERIMENT]
    loso = []
    for source, tr, te in leave_one_source_out(df):
        fold_split = split_df.assign(split=np.where(split_df["source"] == source, "test",
                                     np.where(split_df["split"] == "valid", "valid", "train")))
        y = yolo.export_yolo(images, boxes, fold_split, WORK / "yolo_loso" / source)
        w = train_or_resume(y, RUNS / f"{LOSO_EXPERIMENT}_loso", source, cfg)
        fold_valid = df[df["uid"].isin(fold_split.loc[fold_split["split"] == "valid", "uid"])]
        b = yolo.predict(w, pd.concat([fold_valid, te]))
        t = yolo.tune_conf(b[b["uid"].isin(fold_valid["uid"])], fold_valid)
        b = b[(b["conf"] >= t) & b["uid"].isin(te["uid"])]
        counted = with_counts(te, b)
        evaluate(LOSO_EXPERIMENT, "leave-one-session-out", source, counted, b, conf=t)
        loso.append((counted, b))
    evaluate(LOSO_EXPERIMENT, "leave-one-session-out", "pooled",
             pd.concat([c for c, _ in loso]), pd.concat([b for _, b in loso]))
    res = pd.concat([pd.read_csv(out_csv), pd.DataFrame(results[-6:])])
    res = res.drop_duplicates(["method", "level", "fold"], keep="last")
    res.to_csv(out_csv, index=False)
    display(res[res["level"] == "leave-one-session-out"].set_index(["method", "fold"])[cols].round(3))

## Inference time per image

On this Colab GPU, excluding model loading, for each trained run.

In [ ]:
sample = test.head(20)
timing = {}
for name, w in weights.items():
    yolo.predict(w, sample.head(1))  # warm-up
    t0 = time.perf_counter()
    yolo.predict(w, sample)
    timing[name] = (time.perf_counter() - t0) / len(sample)
pd.Series(timing, name="seconds per image").round(4)

## Next

- Pick the best run by **test MAE and leave-one-session-out**, not mAP alone, and note it in `notes/plan.md`.
- Weights to keep are at `MyDrive/cell-image-counting/outputs/yolo/<name>/weights/best.pt`. Download them to your local `outputs/yolo/` to use them in the local notebooks.
- Still open in Phase 3: SAHI tiling for larger images and the usable-area mask (§5.1).